In [1]:
# bibliotecas
import pandas as pd                                 # manipulação dos dados
import numpy as np                                  # suporte de calculos
from sklearn.preprocessing import LabelEncoder      # encoder de categorico para numero via label
from sklearn.preprocessing import OneHotEncoder     # encoder de categorico para numero via one-hot
from sklearn.preprocessing import TargetEncoder     # encoder de categorico para numero via target
from scipy.spatial.distance import cdist            #


In [2]:
# configurações
pd.set_option('display.float_format', lambda x: f'{x:,.2f}') # demonstra numeros no pandas em float

# pré processamento

Realizar as decisões e praticas encontradas durante o processo de construção do notebook 1_eda

**Remover**
- [x] `CONFIRMADO`Descartar merchant (redundante com category) aos invés de  criar target com 481 opções, somente um one-hot enconder com 14 já mantem as informações
- [x] `CONFIRMADO` Descartar gender (capturado por job)
- [x] `CONFIRMADO` Remover unix_time
- [x] `confirmado` remover colunas identificadoras

**Encoding:**
- [x] `CONFIRMADO` job com Target Encoding (481 categorias)
- [x] `CONFIRMADO` state — One-Hot (51 categorias, agrupar DE e RI em outros)
- [x] `CONFIRMADO` category — One-Hot (14 categorias)


**Transformar**
- [x] `CONFIRMADO` amt — log1p
- [x] `CONFIRMADO` city_pop — log1p


**Criar features novas:**
- [x] `CONFIRMADO` Distancia_km, Calcular distancia entre cliente e merchant via lat/long
- [x] `CONFIRMADO`Extrair hora, dia da semana e mes do trans_date_trans_time
- [x] `CONFIRMADO` dob, calcular a idade do clientes


In [3]:
#df_train = pd.read_csv('fraudTrain.csv', index_col=0)
#df_test = pd.read_csv('fraudTest.csv', index_col=0)

In [4]:
# carrega os dados originais
df_train = pd.read_csv('../data/raw/fraudTrain.csv', index_col=0)
df_test = pd.read_csv('../data/raw/fraudTest.csv', index_col=0)


In [5]:
# verifica linhas e colunas
print(f'Train: {df_train.shape}')
print(f'Test:  {df_test.shape}')

Train: (1296675, 22)
Test:  (555719, 22)


In [6]:
print('is_fraud' in df_test.columns)
print('is_fraud' in df_train.columns)

True
True


In [7]:
# drop das colunas descartadas
df_train.drop(columns=['merchant', 'gender', 'unix_time' , 'street', 'city', 'first', 'last', 'trans_num', 'cc_num', 'zip'], inplace=True)
df_test.drop(columns=['merchant', 'gender', 'unix_time', 'street', 'city', 'first', 'last', 'trans_num', 'cc_num', 'zip'], inplace=True)

In [8]:
# separa o target
y_train = df_train['is_fraud']
y_test = df_test['is_fraud']
#drop do trtaget para evitar data leak
df_train = df_train.drop(columns='is_fraud')
df_test  = df_test.drop(columns='is_fraud')

# confirma
print(f'is_fraud no train: {"is_fraud" in df_train.columns}')
print(f'is_fraud no test:  {"is_fraud" in df_test.columns}')
print(f'Shape train: {df_train.shape}')
print(f'Shape test:  {df_test.shape}')

is_fraud no train: False
is_fraud no test:  False
Shape train: (1296675, 11)
Shape test:  (555719, 11)


In [9]:
# transforma trans_date_trans_time em data e hora, train e test
df_train['trans_date_trans_time'] = pd.to_datetime(df_train['trans_date_trans_time'], format='mixed')
df_test['trans_date_trans_time'] = pd.to_datetime(df_test['trans_date_trans_time'], format='mixed')

# criação das features de tempo train
df_train['hora'] = df_train['trans_date_trans_time'].dt.hour
df_train['dia_semana'] = df_train['trans_date_trans_time'].dt.dayofweek
df_train['mes'] = df_train['trans_date_trans_time'].dt.month

# criação das features de tempo test
df_test['hora'] = df_test['trans_date_trans_time'].dt.hour
df_test['dia_semana'] = df_test['trans_date_trans_time'].dt.dayofweek
df_test['mes'] = df_test['trans_date_trans_time'].dt.month

# calcula idade pelo dob train
df_train['dob'] = pd.to_datetime(df_train['dob'], format='mixed')
df_train['idade'] = (df_train['trans_date_trans_time'] - df_train['dob']).dt.days // 365

# calcula idade pelo dob test
df_test['dob'] = pd.to_datetime(df_test['dob'], format='mixed')
df_test['idade'] = (df_test['trans_date_trans_time'] - df_test['dob']).dt.days // 365

#drop das trans_date_trans_time
df_train.drop(columns=['trans_date_trans_time','dob'], inplace=True)
df_test.drop(columns=['trans_date_trans_time','dob'], inplace=True)

display(df_train[['hora', 'dia_semana', 'mes', 'idade']])
display(df_test[['hora', 'dia_semana', 'mes', 'idade']])


,hora,dia_semana,mes,idade
0,0,1,1,30
1,0,1,1,40
2,0,1,1,56
3,0,1,1,52
4,0,1,1,32
...,...,...,...,...
1296670,12,6,6,58
1296671,12,6,6,40
1296672,12,6,6,52
1296673,12,6,6,39


,hora,dia_semana,mes,idade
0,12,6,6,52
1,12,6,6,30
2,12,6,6,49
3,12,6,6,32
4,12,6,6,65
...,...,...,...,...
555714,23,3,12,54
555715,23,3,12,21
555716,23,3,12,39
555717,23,3,12,55


In [10]:

def haversine(lat1, lon1, lat2, lon2):
    R = 6371 # raio da Terra em km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2]) # graus para radianos
    dlat = lat2 - lat1 # diferenca entre os pontos
    dlon = lon2 - lon1 # diferenca entre os pontos
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2 # calcula dois pontos divididos na esfera
    c = 2 * np.arcsin(np.sqrt(a))  # angulo central entro pontos
    return R * c # multiplica o angulo central pelo raio

# aplica no train
df_train['distancia_km'] = haversine(
    df_train['lat'], df_train['long'],
    df_train['merch_lat'], df_train['merch_long']
)

# aplica no test
df_test['distancia_km'] = haversine(
    df_test['lat'], df_test['long'],
    df_test['merch_lat'], df_test['merch_long']
)

# descarta as 4 colunas originais
df_train = df_train.drop(columns=['lat', 'long', 'merch_lat', 'merch_long'])
df_test  = df_test.drop(columns=['lat', 'long', 'merch_lat', 'merch_long'])

print(df_train['distancia_km'].describe())

count   1,296,675.00
mean           76.11
std            29.12
min             0.02
25%            55.33
50%            78.23
75%            98.50
max           152.12
Name: distancia_km, dtype: float64


In [11]:
print(df_train['distancia_km'].isnull().sum())
print(df_train['distancia_km'].isnull().mean() * 100)

0
0.0


In [12]:
# transformação logaritmica de amt e city_pop no train
df_train['amt'] = np.log1p(df_train['amt'])
df_train['city_pop'] = np.log1p(df_train['city_pop'])

# transformação logaritmica de amt e city_pop no test
df_test['amt'] = np.log1p(df_test['amt'])
df_test['city_pop'] = np.log1p(df_test['city_pop'])

In [13]:
print(df_train[['amt', 'city_pop']].describe())

               amt     city_pop
count 1,296,675.00 1,296,675.00
mean          3.53         8.36
std           1.29         2.45
min           0.69         3.18
25%           2.37         6.61
50%           3.88         7.81
75%           4.43         9.92
max          10.27        14.88


In [14]:
# enconding das variaveris categoricas
# agrupamento de DE e RI em 'Outros'
df_train['state'] = df_train['state'].replace(['DE', 'RI'], 'Outros')
df_test['state'] = df_test['state'].replace(['DE', 'RI'], 'Outros')

# onehotencoder do 'state'
df_train = pd.get_dummies(df_train, columns=['state'], prefix=['state'], drop_first=False, dtype=int)
df_test = pd.get_dummies(df_test, columns=['state'], prefix=['state'], drop_first=False, dtype=int)

# garante que train e test tem as mesmas colunas de state
cols_train = set(df_train.filter(like='state_').columns)
cols_test  = set(df_test.filter(like='state_').columns)
print(f'Colunas diferentes: {cols_train.symmetric_difference(cols_test)}')

Colunas diferentes: set()


In [15]:
# onehotencoder do 'category'
df_train = pd.get_dummies(df_train, columns=['category'], prefix=['cat'], drop_first=False, dtype=int)
df_test = pd.get_dummies(df_test, columns=['category'], prefix=['cat'], drop_first=False, dtype=int)

# garante que train e test tem as mesmas colunas de state
cols_train = set(df_train.filter(like='cat_').columns)
cols_test  = set(df_test.filter(like='cat_').columns)
print(f'Colunas diferentes: {cols_train.symmetric_difference(cols_test)}')

Colunas diferentes: set()


In [16]:
# target enconder do job

target = TargetEncoder(target_type='binary', smooth='auto')

# fit só no train com o target
df_train['job_encoded'] = target.fit_transform(df_train[['job']], y_train).ravel()

# transform no test usando o que aprendeu no train
df_test['job_encoded'] = target.transform(df_test[['job']]).ravel()

# descarta a coluna original
df_train = df_train.drop(columns='job')
df_test  = df_test.drop(columns='job')

print(df_train['job_encoded'].describe())

count   1,296,675.00
mean            0.01
std             0.01
min             0.00
25%             0.00
50%             0.01
75%             0.01
max             1.00
Name: job_encoded, dtype: float64


In [17]:
print(df_train.columns.tolist())
print(df_train.shape)

['amt', 'city_pop', 'hora', 'dia_semana', 'mes', 'idade', 'distancia_km', 'state_AK', 'state_AL', 'state_AR', 'state_AZ', 'state_CA', 'state_CO', 'state_CT', 'state_DC', 'state_FL', 'state_GA', 'state_HI', 'state_IA', 'state_ID', 'state_IL', 'state_IN', 'state_KS', 'state_KY', 'state_LA', 'state_MA', 'state_MD', 'state_ME', 'state_MI', 'state_MN', 'state_MO', 'state_MS', 'state_MT', 'state_NC', 'state_ND', 'state_NE', 'state_NH', 'state_NJ', 'state_NM', 'state_NV', 'state_NY', 'state_OH', 'state_OK', 'state_OR', 'state_Outros', 'state_PA', 'state_SC', 'state_SD', 'state_TN', 'state_TX', 'state_UT', 'state_VA', 'state_VT', 'state_WA', 'state_WI', 'state_WV', 'state_WY', 'cat_entertainment', 'cat_food_dining', 'cat_gas_transport', 'cat_grocery_net', 'cat_grocery_pos', 'cat_health_fitness', 'cat_home', 'cat_kids_pets', 'cat_misc_net', 'cat_misc_pos', 'cat_personal_care', 'cat_shopping_net', 'cat_shopping_pos', 'cat_travel', 'job_encoded']
(1296675, 72)


In [18]:
# confere linhas e colunas finais
print(f'Shape train: {df_train.shape}')
print(f'Shape test:  {df_test.shape}')

Shape train: (1296675, 72)
Shape test:  (555719, 72)


In [19]:
# verifica se tem nulos em alguma coluna
print('Nulos no train:')
print(df_train.isnull().sum()[df_train.isnull().sum() > 0])

print('\nNulos no test:')
print(df_test.isnull().sum()[df_test.isnull().sum() > 0])

# verifica tipos
print('\nTipos:')
print(df_train.dtypes.value_counts())

Nulos no train:
Series([], dtype: int64)

Nulos no test:
Series([], dtype: int64)

Tipos:
int64      65
float64     4
int32       3
Name: count, dtype: int64


In [20]:
# salva os dados processados
df_train.to_csv('../data/processed/X_train.csv', index=False)
df_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Dados salvos em data/processed/')

Dados salvos em data/processed/
